In [69]:
#import tmdb API key

tmdb_api_key = "c288cfd4a1d527ed2e6d111d6d23a5ae"

import requests

api_key = tmdb_api_key 
url = f"https://api.themoviedb.org/3/movie/550?api_key={api_key}"

response = requests.get(url)
data = response.json()

print(data)

{'adult': False, 'backdrop_path': '/xRyINp9KfMLVjRiO5nCsoRDdvvF.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'https://www.20thcenturystudios.com/movies/fight-club', 'id': 550, 'imdb_id': 'tt0137523', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Fight Club', 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.', 'popularity': 26.1054, 'poster_path': '/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg', 'production_companies': [{'id': 711, 'logo_path': '/tEiIH5QesdheJmDAqQwvtN60727.png', 'name': 'Fox 2000 Pictures', 'origin_country': 'US'}, {'id': 508, 'logo_path': '/4sGWXoboEkWPphI6es6rTmqkCBh.png', 'name': 'Regency Enterprises', '

In [70]:
#import libraries
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from pandas import json_normalize 
import networkx as nx
import datetime

In [71]:
#Test
query = "https://api.themoviedb.org/3/movie/" + "438631" + "/credits?api_key=" + api_key + "&language=en-US"
response =  requests.get(query)
array = response.json()
temp_cast = json_normalize(array, 'cast')
temp_cast.head()

,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,cast_id,character,credit_id,order
0,False,2,1190668,Acting,Timothée Chalamet,Timothée Chalamet,8.7623,/axENiFIrSz5B7UuWkMT7PDe7CaO.jpg,13,Paul Atreides,5b4d01bac3a36823d803cd45,0
1,False,1,933238,Acting,Rebecca Ferguson,Rebecca Ferguson,7.6158,/lJloTOheuQSirSLXNA3JHsrMNfH.jpg,14,Lady Jessica Atreides,5b90742fc3a368222e002f41,1
2,False,2,25072,Acting,Oscar Isaac,Oscar Isaac,5.3077,/dW5U5yrIIPmMjRThR9KT2xH6nTz.jpg,53,Duke Leto Atreides,5c50bc070e0a2612cccedcb3,2
3,False,2,117642,Acting,Jason Momoa,Jason Momoa,7.2932,/3troAR6QbSb6nUFMDu61YCCWLKa.jpg,75,Duncan Idaho,5c6610319251412fc6017577,3
4,False,2,1640,Acting,Stellan Skarsgård,Stellan Skarsgård,4.4146,/x78BtYHElirO7Iw8bL4m8CnzRDc.jpg,31,Baron Vladimir Harkonnen,5c364b61c3a368273c1c5fee,4


In [82]:
#Read csv
movies = pd.read_csv("films_tmdb.csv")
movies = movies.drop_duplicates()
movies.to_csv('cleaned_output.csv', index=False)

In [97]:
#new csv with film details
import csv
import requests
import time

BASE_URL = "https://api.themoviedb.org/3"
API_KEY = "c288cfd4a1d527ed2e6d111d6d23a5ae"

def get_movie_credits(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}/credits"
    params = {"api_key": API_KEY}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error getting credits for {movie_id}: {response.status_code}")
        print(response.text)
        return None
    return response.json()

def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error getting details for {movie_id}: {response.status_code}")
        print(response.text)
        return None
    return response.json()

def extract_director(crew_list):
    directors = [c["name"] for c in crew_list if c["job"] == "Director"]
    return ", ".join(directors)

def extract_top_cast(cast_list, n=15):
    return ", ".join([c["name"] for c in cast_list[:n]])

def extract_genres(details):
    if "genres" in details:
        return ", ".join([g["name"] for g in details["genres"]])
    return ""

def process_csv(input_file, output_file):
    with open(input_file, newline="", encoding="utf-8") as infile, \
         open(output_file, "w", newline="", encoding="utf-8") as outfile:

        reader = csv.DictReader(infile)
        fieldnames = ["tmdb_id", "title", "director", "top_cast", "genres", "release_date", "budget"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            movie_id = row["id"].strip()
            if not movie_id:
                continue

            # Fetch data
            credits = get_movie_credits(movie_id)
            details = get_movie_details(movie_id)

            if not credits or not details:
                print(f"⚠️ Skipping movie ID {movie_id}")
                continue

            # Extract info
            title = details.get("title", "")
            director = extract_director(credits["crew"])
            cast = extract_top_cast(credits["cast"])
            genres = extract_genres(details)
            release_date = details.get("release_date", "")
            budget = details.get("budget", "")

            # Write to CSV
            writer.writerow({
                "tmdb_id": movie_id,
                "title": title,
                "director": director,
                "top_cast": cast,
                "genres": genres,
                "release_date": release_date,
                "budget": budget
            })

            time.sleep(0.25)  # avoid rate limits

if __name__ == "__main__":
    process_csv("cleaned_output.csv", "film_details.csv")

Error getting credits for 35430: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
Error getting details for 35430: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
⚠️ Skipping movie ID 35430
Error getting credits for 103783: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
Error getting details for 103783: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
⚠️ Skipping movie ID 103783


In [98]:
#new csv with film details with corrected movie ID
import csv
import requests
import time

BASE_URL = "https://api.themoviedb.org/3"
API_KEY = "c288cfd4a1d527ed2e6d111d6d23a5ae"

def get_movie_credits(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}/credits"
    params = {"api_key": API_KEY}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error getting credits for {movie_id}: {response.status_code}")
        print(response.text)
        return None
    return response.json()

def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error getting details for {movie_id}: {response.status_code}")
        print(response.text)
        return None
    return response.json()

def extract_director(crew_list):
    directors = [c["name"] for c in crew_list if c["job"] == "Director"]
    return ", ".join(directors)

def extract_top_cast(cast_list, n=15):
    return ", ".join([c["name"] for c in cast_list[:n]])

def extract_genres(details):
    if "genres" in details:
        return ", ".join([g["name"] for g in details["genres"]])
    return ""

def process_csv(input_file, output_file):
    with open(input_file, newline="", encoding="utf-8") as infile, \
         open(output_file, "w", newline="", encoding="utf-8") as outfile:

        reader = csv.DictReader(infile)
        fieldnames = ["tmdb_id", "title", "director", "top_cast", "genres", "release_date", "budget"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            movie_id = row["id"].strip()
            if not movie_id:
                continue

            # Fetch data
            credits = get_movie_credits(movie_id)
            details = get_movie_details(movie_id)

            if not credits or not details:
                print(f"⚠️ Skipping movie ID {movie_id}")
                continue

            # Extract info
            title = details.get("title", "")
            director = extract_director(credits["crew"])
            cast = extract_top_cast(credits["cast"])
            genres = extract_genres(details)
            release_date = details.get("release_date", "")
            budget = details.get("budget", "")

            # Write to CSV
            writer.writerow({
                "tmdb_id": movie_id,
                "title": title,
                "director": director,
                "top_cast": cast,
                "genres": genres,
                "release_date": release_date,
                "budget": budget
            })

            time.sleep(0.25)  # avoid rate limits

if __name__ == "__main__":
    process_csv("cleaned_output_v2.csv", "film_details_v2.csv")

Error getting credits for 35430: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
Error getting details for 35430: 404
{"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
⚠️ Skipping movie ID 35430


In [5]:
# Load your CSV
df = pd.read_csv("film_details_v2.csv")

# REMOVE commas after suffixes like "Jr.," 
df['top_cast_clean'] = df['top_cast'].str.replace(
    r'(Jr\.|Sr\.|III|IV|Ph\.D\.)\s*,',
    r'\1',  # remove the comma
    regex=True
)

# Split the column into multiple columns
cast_split = df['top_cast_clean'].str.split(',', expand=True)

# Strip whitespace from each entry
cast_split = cast_split.apply(lambda col: col.str.strip())

# Rename columns dynamically
cast_split.columns = [f"Cast {i+1}" for i in range(cast_split.shape[1])]

# Combine back with original dataframe
df = df.drop(columns=['top_cast', 'top_cast_clean']).join(cast_split)

# Save result (optional)
df.to_csv("film_details_v3.csv", index=False)